# 📊 Project 10.1 — Sorting Algorithm Benchmark
**DSA for Mechatronics · Week 10 — Sorting Algorithms**

> **Run all cells top to bottom. Submit the .ipynb on Blackboard.**
> The final cell prints your personalised report — be ready to explain every step.


In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║            ENTER YOUR STUDENT ID BELOW              ║
# ╚══════════════════════════════════════════════════════╝

STUDENT_ID = "12345678"   # ← replace with your real student ID

import random, hashlib, time
from copy import deepcopy
_seed = int(hashlib.md5(STUDENT_ID.encode()).hexdigest(), 16) % (2**31)
random.seed(_seed)
print(f"Student ID  : {STUDENT_ID}")
print(f"Dataset seed: {_seed}")
print("✅ Your personal dataset is ready.")


## The Scenario
A test bench records thousands of sensor readings that must be sorted before analysis.
You will implement six sorting algorithms from scratch, run them all on the **same
personalised dataset**, count comparisons and swaps, and rank them by speed.

Algorithms: **Bubble · Selection · Insertion · Merge · Quick · Heap**


## Step 1 — Generate the benchmark dataset

In [ ]:
# ── Personalised parameters ──────────────────────────────────────
N_SMALL   = random.randint(20, 30)     # for O(n²) sorts — keep it small
N_LARGE   = random.randint(800, 1200)  # for O(n log n) sorts
VAL_RANGE = random.randint(200, 500)

# Small dataset (used for all 6 algorithms + step tracing)
small_data = [random.randint(1, VAL_RANGE) for _ in range(N_SMALL)]
# Large dataset (used for O(n log n) only — O(n²) would be very slow)
large_data = [random.randint(1, VAL_RANGE) for _ in range(N_LARGE)]

# Nearly-sorted variant (10% shuffled) — shows insertion sort advantage
nearly_sorted = sorted(small_data)
n_swaps = max(1, N_SMALL // 10)
for _ in range(n_swaps):
    i, j = random.sample(range(N_SMALL), 2)
    nearly_sorted[i], nearly_sorted[j] = nearly_sorted[j], nearly_sorted[i]

print(f"Benchmark dataset parameters:")
print(f"  Small dataset  : {N_SMALL} values  (range 1–{VAL_RANGE})")
print(f"  Large dataset  : {N_LARGE} values  (range 1–{VAL_RANGE})")
print(f"  Nearly-sorted  : {N_SMALL} values  ({n_swaps} swap(s) from sorted)")
print()
print(f"  Small dataset  : {small_data}")
print(f"  Nearly sorted  : {nearly_sorted}")


## Step 2 — Implement all six sorting algorithms

In [ ]:
# Each sort returns (sorted_list, comparisons, swaps/moves)

def bubble_sort(arr):
    """
    Bubble sort: repeatedly swap adjacent elements if out of order.
    After pass i, the i largest elements are in their final position.
    O(n²) comparisons and swaps. Stable. Best case O(n) with early exit.
    """
    a = arr[:]
    n = len(a)
    comps = swaps = 0
    for i in range(n):
        swapped = False
        for j in range(0, n - i - 1):
            comps += 1
            if a[j] > a[j + 1]:
                a[j], a[j + 1] = a[j + 1], a[j]
                swaps += 1
                swapped = True
        if not swapped:
            break   # early exit if already sorted
    return a, comps, swaps

def selection_sort(arr):
    """
    Selection sort: find minimum of remaining, place it at front.
    Always O(n²) comparisons, O(n) swaps. Unstable. No early exit.
    """
    a = arr[:]
    n = len(a)
    comps = swaps = 0
    for i in range(n):
        min_idx = i
        for j in range(i + 1, n):
            comps += 1
            if a[j] < a[min_idx]:
                min_idx = j
        if min_idx != i:
            a[i], a[min_idx] = a[min_idx], a[i]
            swaps += 1
    return a, comps, swaps

def insertion_sort(arr):
    """
    Insertion sort: insert each element into its correct position
    in the already-sorted prefix. O(n²) worst case, O(n) best case.
    Stable. Excellent for nearly-sorted data.
    """
    a = arr[:]
    comps = moves = 0
    for i in range(1, len(a)):
        key = a[i]
        j = i - 1
        while j >= 0:
            comps += 1
            if a[j] > key:
                a[j + 1] = a[j]
                moves += 1
                j -= 1
            else:
                break
        a[j + 1] = key
    return a, comps, moves

def merge_sort(arr):
    """
    Merge sort: divide array in half, sort each half, merge.
    Always O(n log n) comparisons. Stable. Uses O(n) extra space.
    """
    comps = [0]
    def merge(left, right):
        result = []
        i = j = 0
        while i < len(left) and j < len(right):
            comps[0] += 1
            if left[i] <= right[j]:
                result.append(left[i]); i += 1
            else:
                result.append(right[j]); j += 1
        result.extend(left[i:]); result.extend(right[j:])
        return result
    def sort(a):
        if len(a) <= 1: return a
        mid = len(a) // 2
        return merge(sort(a[:mid]), sort(a[mid:]))
    return sort(arr[:]), comps[0], 0   # 0 in-place swaps

def quick_sort(arr):
    """
    Quick sort: partition around pivot (median-of-3 pivot selection),
    recursively sort sub-arrays. Average O(n log n). Unstable. In-place.
    Median-of-3 avoids O(n²) worst case on sorted input.
    """
    a = arr[:]
    comps = [0]; swaps = [0]
    def median3(lo, hi):
        mid = (lo + hi) // 2
        comps[0] += 2
        if a[lo] > a[mid]: a[lo], a[mid] = a[mid], a[lo]; swaps[0] += 1
        if a[lo] > a[hi]:  a[lo], a[hi]  = a[hi],  a[lo]; swaps[0] += 1
        if a[mid] > a[hi]: a[mid], a[hi] = a[hi], a[mid]; swaps[0] += 1
        return a[mid]   # median is now at mid
    def partition(lo, hi):
        pivot = median3(lo, hi)
        i, j = lo - 1, hi
        while True:
            i += 1
            while i < hi:
                comps[0] += 1
                if a[i] >= pivot: break
                i += 1
            j -= 1
            while j > lo:
                comps[0] += 1
                if a[j] <= pivot: break
                j -= 1
            if i >= j: break
            a[i], a[j] = a[j], a[i]; swaps[0] += 1
        a[i], a[hi] = a[hi], a[i]; swaps[0] += 1
        return i
    def sort(lo, hi):
        if lo < hi:
            p = partition(lo, hi)
            sort(lo, p - 1)
            sort(p + 1, hi)
    import sys; sys.setrecursionlimit(10000)
    if len(a) > 1: sort(0, len(a) - 1)
    return a, comps[0], swaps[0]

def heap_sort(arr):
    """
    Heap sort: build max-heap (O(n)), then repeatedly extract max (O(n log n)).
    Always O(n log n). Unstable. In-place. No extra space.
    """
    a = arr[:]
    n = len(a)
    comps = [0]; swaps = [0]
    def sift_down(root, end):
        while True:
            child = 2 * root + 1
            if child > end: break
            if child + 1 <= end:
                comps[0] += 1
                if a[child] < a[child + 1]: child += 1
            comps[0] += 1
            if a[root] < a[child]:
                a[root], a[child] = a[child], a[root]
                swaps[0] += 1
                root = child
            else:
                break
    # Build heap
    for i in range(n // 2 - 1, -1, -1):
        sift_down(i, n - 1)
    # Sort
    for end in range(n - 1, 0, -1):
        a[0], a[end] = a[end], a[0]; swaps[0] += 1
        sift_down(0, end - 1)
    return a, comps[0], swaps[0]

print("All six sorting algorithms implemented.")
print("  ✅ bubble_sort")
print("  ✅ selection_sort")
print("  ✅ insertion_sort")
print("  ✅ merge_sort")
print("  ✅ quick_sort")
print("  ✅ heap_sort")


## Step 3 — Run all sorts on small dataset + count operations

In [ ]:
SORTS = [
    ("Bubble",    bubble_sort),
    ("Selection", selection_sort),
    ("Insertion", insertion_sort),
    ("Merge",     merge_sort),
    ("Quick",     quick_sort),
    ("Heap",      heap_sort),
]

results_small = {}
print(f"Results on small dataset (n={N_SMALL}):")
print(f"  {'Algorithm':<12}  {'Comparisons':>13}  {'Swaps/Moves':>12}  {'Correct':>8}  Time (ms)")
print(f"  {'─'*12}  {'─'*13}  {'─'*12}  {'─'*8}  {'─'*10}")
expected = sorted(small_data)
for name, fn in SORTS:
    t0 = time.perf_counter()
    result, comps, ops = fn(small_data)
    elapsed_ms = (time.perf_counter() - t0) * 1000
    correct = result == expected
    results_small[name] = (comps, ops, correct, elapsed_ms)
    print(f"  {name:<12}  {comps:13,d}  {ops:12,d}  {'✅' if correct else '❌':>8}  {elapsed_ms:.3f}")

# Nearly-sorted comparison: bubble vs insertion
_, bs_ns_comps, bs_ns_swaps = bubble_sort(nearly_sorted)
_, is_ns_comps, is_ns_moves = insertion_sort(nearly_sorted)
print()
print(f"Nearly-sorted comparison (n={N_SMALL}, {n_swaps} swap(s) from sorted):")
print(f"  Bubble    : {bs_ns_comps:,} comparisons,  {bs_ns_swaps:,} swaps")
print(f"  Insertion : {is_ns_comps:,} comparisons,  {is_ns_moves:,} moves")
winner_ns = "Insertion" if is_ns_comps <= bs_ns_comps else "Bubble"
print(f"  Advantage : {winner_ns} sort uses fewer comparisons on nearly-sorted data")


## Step 4 — Benchmark O(n log n) sorts on large dataset

In [ ]:
FAST_SORTS = [("Merge", merge_sort), ("Quick", quick_sort), ("Heap", heap_sort)]

results_large = {}
print(f"Benchmark on large dataset (n={N_LARGE}):")
print(f"  {'Algorithm':<12}  {'Comparisons':>13}  {'Swaps/Moves':>12}  {'Correct':>8}  Time (ms)")
print(f"  {'─'*12}  {'─'*13}  {'─'*12}  {'─'*8}  {'─'*10}")
expected_large = sorted(large_data)
for name, fn in FAST_SORTS:
    t0 = time.perf_counter()
    result, comps, ops = fn(large_data)
    elapsed_ms = (time.perf_counter() - t0) * 1000
    correct = result == expected_large
    results_large[name] = (comps, ops, correct, elapsed_ms)
    print(f"  {name:<12}  {comps:13,d}  {ops:12,d}  {'✅' if correct else '❌':>8}  {elapsed_ms:.3f}")

fastest = min(results_large, key=lambda k: results_large[k][3])
most_comps = min(results_large, key=lambda k: results_large[k][0])
print()
print(f"  Fastest (wall time)      : {fastest}")
print(f"  Fewest comparisons       : {most_comps}")

# Python built-in timing
t0 = time.perf_counter()
_ = sorted(large_data)
builtin_ms = (time.perf_counter() - t0) * 1000
print(f"  Python sorted() time     : {builtin_ms:.3f} ms  (Timsort — reference)")
ratio = round(results_large[fastest][3] / builtin_ms, 1) if builtin_ms > 0 else 0
print(f"  Best custom / builtin    : {ratio}× overhead")


## 📋 Final Report

In [ ]:
W = 60
print("╔" + "═"*W + "╗")
print("║" + " SORTING ALGORITHM BENCHMARK — REPORT".center(W) + "║")
print("╠" + "═"*W + "╣")
print(f"║  {'Student ID':<28}: {STUDENT_ID:<28}║")
print(f"║  {'Dataset seed':<28}: {_seed:<28}║")
print("╠" + "═"*W + "╣")
print("║  SMALL DATASET (n=N)  all 6 algorithms".ljust(W+1) + "║")
print(f"║  {'n':<28}: {N_SMALL:<28}║")
for name, (comps, ops, correct, ms) in results_small.items():
    line = f"{name}: {comps:,} comps, {ops:,} ops"
    print(f"║  {line:<58}║")
print("╠" + "═"*W + "╣")
print("║  LARGE DATASET (n=N)  O(n log n) only".ljust(W+1) + "║")
print(f"║  {'n':<28}: {N_LARGE:<28}║")
for name, (comps, ops, correct, ms) in results_large.items():
    line = f"{name}: {comps:,} comps, {ms:.2f}ms"
    print(f"║  {line:<58}║")
print(f"║  {'Fastest algorithm':<28}: {fastest:<28}║")
print(f"║  {'Python built-in (Timsort)':<28}: {builtin_ms:.3f} ms{'':<20}║")
print(f"║  {'Custom/builtin ratio':<28}: {ratio}×{'':<26}║")
print("╠" + "═"*W + "╣")
print("║  NEARLY-SORTED DATA".ljust(W+1) + "║")
print(f"║  {'Winner (fewer comps)':<28}: {winner_ns:<28}║")
print("╚" + "═"*W + "╝")


---
## 📝 Reflection — answer before submitting

Double-click this cell to edit. Write 2–4 sentences for each question.

---

**Q1 — What does your output show?**
Describe the key results from your final report. What does it tell you about sorting in this context?

*Your answer here:*

---

**Q2 — Which sorting concept did you find most important, and why?**
Pick one algorithm or pattern (merge sort, quickselect, interval merging, counting sort…) and explain what problem it solves best.

*Your answer here:*

---

**Q3 — What would change if your student ID changed?**
Which specific numbers or results in the final report would be different, and why?

*Your answer here:*
